# Smart UAE Tourism Agent
This notebook demonstrates the implementation of a LangChain-based tourism agent for the UAE. The agent uses tools to provide information about UAE cities, prayer times, and trip budgets.

In [ ]:
# Import required libraries
import json
from langchain.agents import initialize_agent, AgentType
from langchain.llms import OpenAI
from langchain.tools import Tool
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import os
import requests

# Load environment variables
load_dotenv()

# Load the knowledge base
with open('uae_knowledge.json') as f:
    uae_data = json.load(f)


## Define Tools
The tools include:
- `UAEKnowledgeSearchTool`: Searches the knowledge base for city-specific or general information.
- `PrayerTimeTool`: Fetches real-time prayer times for a given city.
- `TripBudgetPlanner`: Estimates trip costs based on city, days, and travel style.

In [ ]:
# Define UAEKnowledgeSearchTool
class UAEKnowledgeSearchTool:
    def __init__(self, knowledge):
        self.knowledge = knowledge

    def search(self, query):
        query = query.lower()
        for city, data in self.knowledge['cities'].items():
            if city.lower() in query:
                return f"Information about {city}:
" + json.dumps(data, indent=2)
        return "No relevant information found."

# Define PrayerTimeTool
class PrayerTimeTool:
    @staticmethod
    def get_prayer_times(city):
        response = requests.get(f'https://api.aladhan.com/v1/timingsByCity?city={city}&country=UAE')
        if response.status_code == 200:
            data = response.json()
            return data['data']['timings']
        return 'Could not fetch prayer times.'

# Define TripBudgetPlanner
class TripBudgetPlanner:
    RATES = {"budget": 150, "standard": 400, "luxury": 1000}

    @staticmethod
    def estimate(city, days, style):
        style = style.lower()
        if style not in TripBudgetPlanner.RATES:
            return "Invalid style. Choose from: budget, standard, luxury."
        cost = TripBudgetPlanner.RATES[style] * days
        return f"Estimated cost for {days} day(s) in {city} ({style}): {cost} AED."


## Initialize Agent
The agent integrates the tools and uses OpenAI's LLM to process user queries.

In [ ]:
# Initialize tools
knowledge_tool = UAEKnowledgeSearchTool(uae_data)
prayer_tool = PrayerTimeTool()
budget_tool = TripBudgetPlanner()

# Define tools for the agent
tools = [
    Tool(name='Knowledge Search', func=knowledge_tool.search, description='Search UAE knowledge base.'),
    Tool(name='Prayer Times', func=prayer_tool.get_prayer_times, description='Get prayer times for a city.'),
    Tool(name='Trip Budget', func=budget_tool.estimate, description='Estimate trip budget.')
]

# Initialize the agent
llm = OpenAI(openai_api_key=os.getenv('OPENAI_API_KEY'), temperature=0)
agent = initialize_agent(tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=True)


## Example Queries
Test the agent with example queries.

In [ ]:
# Example queries
print(agent.run('What are the top attractions in Dubai?'))
print(agent.run('What are the prayer times in Abu Dhabi?'))
print(agent.run('Estimate a 3-day luxury trip to Ras Al Khaimah.'))